# ORION wake word training recovery - openWakeWord Colab

Notebook de recuperacion basado en la primera version historica de `train_orion_colab.ipynb` (`943606f`), la version que genero correctamente los 5000 clips:

- positive_train: 1500
- positive_test: 1000
- negative_train: 1500
- negative_test: 1000

Aplica solo las correcciones necesarias para reanudar desde Google Drive, normalizar WAV, generar features, entrenar ONNX y exportar `orion_v2.onnx`.

No modifica el pipeline estable de ORION, no usa AudioSet, no convierte a TFLite y no guarda artefactos pesados en Git.

In [ ]:
# 1. Montar Drive
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE_DIR = Path('/content/drive/MyDrive/orion_wakeword_training')
REPOS_DIR = BASE_DIR / 'repos'
DATASETS_DIR = BASE_DIR / 'datasets'
CLIPS_DIR = BASE_DIR / 'clips'
FEATURES_DIR = BASE_DIR / 'features'
MODELS_DIR = BASE_DIR / 'models'
EXPORT_DIR = BASE_DIR / 'exports'
OPENWAKEWORD_DIR = REPOS_DIR / 'openWakeWord'
PIPER_DIR = Path('/content/piper-sample-generator-v2')
FMA_PATH = DATASETS_DIR / 'fma'
MIT_RIRS_PATH = DATASETS_DIR / 'mit_rirs'

for path in [BASE_DIR, REPOS_DIR, DATASETS_DIR, CLIPS_DIR, FEATURES_DIR, MODELS_DIR, EXPORT_DIR, FMA_PATH, MIT_RIRS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print('Directorio persistente:', BASE_DIR)
print('Piper temporal:', PIPER_DIR)

In [ ]:
# 2. Verificar Python/GPU
import os, platform, subprocess, sys
print('Python:', sys.version)
print('Platform:', platform.platform())
!nvidia-smi
assert sys.version_info[:2] == (3, 11), 'Usa runtime Colab Python 3.11'
gpu_name = subprocess.check_output('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True, text=True).strip()
print('GPU:', gpu_name)
assert 'T4' in gpu_name, 'Selecciona GPU T4 en Runtime > Change runtime type'

In [ ]:
# 3. Instalar dependencias necesarias en una sola celda
# No instalar tensorflow-cpu==2.8.1. No se convierte a TFLite.
%pip install -q --upgrade pip
%pip install -q \
  piper-phonemize==1.1.0 \
  webrtcvad \
  torchinfo==1.8.0 \
  torchmetrics==1.2.0 \
  lightning-utilities \
  speechbrain==0.5.14 \
  audiomentations==0.33.0 \
  torch-audiomentations==0.11.0 \
  acoustics==0.2.6 \
  pronouncing==0.2.0 \
  deep-phonemizer==0.0.19 \
  mutagen==1.47.0 \
  librosa==0.10.2.post1 \
  soxr==0.5.0.post1 \
  onnxruntime==1.27.0 \
  onnx==1.16.1 \
  ai-edge-litert==2.1.5 \
  speexdsp-ns==0.1.2 \
  fsspec==2023.6.0 \
  datasets==2.14.4

In [ ]:
# 4. Clonar repositorios compatibles
import os, shutil, subprocess, sys

PIPER_MODEL_DRIVE = MODELS_DIR / 'piper' / 'en_US-libritts_r-medium.pt'
PIPER_CONFIG_DRIVE = MODELS_DIR / 'piper' / 'en_US-libritts_r-medium.pt.json'
PIPER_MODEL_TEMP = PIPER_DIR / 'models' / 'en_US-libritts_r-medium.pt'
PIPER_CONFIG_TEMP = PIPER_DIR / 'models' / 'en_US-libritts_r-medium.pt.json'

def run(cmd, cwd=None, env=None):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], cwd=cwd, env=env, check=True)

if not OPENWAKEWORD_DIR.exists():
    run(['git', 'clone', 'https://github.com/dscripka/openWakeWord.git', OPENWAKEWORD_DIR])
else:
    print('Reutilizando', OPENWAKEWORD_DIR)

if PIPER_DIR.exists():
    shutil.rmtree(PIPER_DIR)
run(['git', 'clone', '--branch', 'v2.0.0', '--depth', '1', 'https://github.com/rhasspy/piper-sample-generator.git', PIPER_DIR])
(PIPER_DIR / 'models').mkdir(parents=True, exist_ok=True)

missing_piper = [str(path) for path in [PIPER_MODEL_DRIVE, PIPER_CONFIG_DRIVE] if not path.exists() or path.stat().st_size == 0]
if missing_piper:
    raise FileNotFoundError('Falta el modelo Piper persistente en Drive:\n' + '\n'.join(missing_piper))

shutil.copy2(PIPER_MODEL_DRIVE, PIPER_MODEL_TEMP)
shutil.copy2(PIPER_CONFIG_DRIVE, PIPER_CONFIG_TEMP)
print('Piper temporal listo:', PIPER_DIR)

os.environ['PYTHONPATH'] = f"{OPENWAKEWORD_DIR}:{PIPER_DIR}:" + os.environ.get('PYTHONPATH', '')
print('PYTHONPATH=', os.environ['PYTHONPATH'])

In [ ]:
# 5. Aplicar parches de compatibilidad
import importlib.util, re, site


def patch_torch_load_file(path):
    path = Path(path)
    text = path.read_text(encoding='utf-8')
    patched = re.sub(
        r"torch\.load\((checkpoint_path|model_path|voice_path)(?!,\s*weights_only=)",
        r"torch.load(, weights_only=False",
        text,
    )
    if patched != text:
        path.write_text(patched, encoding='utf-8')
        print('Patched torch.load:', path)
    else:
        print('torch.load OK:', path)

patch_torch_load_file(PIPER_DIR / 'generate_samples.py')

spec = importlib.util.find_spec('dp') or importlib.util.find_spec('deep_phonemizer')
if spec and spec.origin:
    dp_root = Path(spec.origin).resolve().parent
    for py_file in dp_root.rglob('*.py'):
        source = py_file.read_text(encoding='utf-8', errors='ignore')
        if 'torch.load(' in source:
            patch_torch_load_file(py_file)
else:
    print('deep-phonemizer package not found for patch scan')

# openWakeWord data.py: despues de convertir mixed_clips_batch a NumPy, debe usar axis=1.
data_py = OPENWAKEWORD_DIR / 'openwakeword' / 'data.py'
data_text = data_py.read_text(encoding='utf-8')
data_patched = data_text.replace(
    'np.where(mixed_clips_batch.max(dim=1) != 0)[0]',
    'np.where(mixed_clips_batch.max(axis=1) != 0)[0]',
)
if data_patched != data_text:
    data_py.write_text(data_patched, encoding='utf-8')
    print('Patched openwakeword/data.py max(axis=1)')
else:
    print('openwakeword/data.py max(axis=1) ya estaba aplicado')

# openWakeWord train.py exporta ONNX y luego intenta TFLite. Esta fase es ONNX-only.
train_py = OPENWAKEWORD_DIR / 'openwakeword' / 'train.py'
train_text = train_py.read_text(encoding='utf-8')
tflite_old = """        # Convert the model from onnx to tflite format
        convert_onnx_to_tflite(os.path.join(config["output_dir"], config["model_name"] + ".onnx"),
                               os.path.join(config["output_dir"], config["model_name"] + ".tflite"))"""
tflite_new = """        # ORION patch: ONNX-only training. Do not convert to TFLite.
        logging.info('ORION patch: skipping TFLite conversion; ONNX export only.')"""
if tflite_new in train_text:
    print('TFLite patch ya estaba aplicado')
elif tflite_old in train_text:
    train_py.write_text(train_text.replace(tflite_old, tflite_new, 1), encoding='utf-8')
    print('Patched train.py ONNX-only')
else:
    raise RuntimeError('No se encontro el bloque TFLite esperado en openwakeword/train.py')

print('Parches aplicados')

In [ ]:
# 6. Descargar modelos auxiliares y datasets
import json, shutil, urllib.request, zipfile


def download(url, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print('Reutilizando', dest)
        return dest
    print('Descargando', url, '->', dest)
    urllib.request.urlretrieve(url, dest)
    if not dest.exists() or dest.stat().st_size == 0:
        raise RuntimeError(f'Descarga vacia: {dest}')
    return dest


def convert_audio(src, dest):
    src = Path(src)
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix('.tmp.wav')
    run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-i', src, '-ac', '1', '-ar', '16000', '-sample_fmt', 's16', tmp])
    tmp.replace(dest)


def is_pcm16_mono_16k(path):
    try:
        raw = subprocess.check_output([
            'ffprobe', '-v', 'error', '-select_streams', 'a:0',
            '-show_entries', 'stream=codec_name,sample_rate,channels',
            '-of', 'json', str(path),
        ], text=True)
        stream = json.loads(raw).get('streams', [{}])[0]
        return stream.get('codec_name') == 'pcm_s16le' and str(stream.get('sample_rate')) == '16000' and int(stream.get('channels', 0)) == 1
    except Exception:
        return False

if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
    raise RuntimeError('Faltan ffmpeg/ffprobe en el runtime de Colab')

OWW_RESOURCES = OPENWAKEWORD_DIR / 'openwakeword' / 'resources' / 'models'
OWW_RESOURCES.mkdir(parents=True, exist_ok=True)
BASE_RELEASE = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
download(f'{BASE_RELEASE}/melspectrogram.onnx', OWW_RESOURCES / 'melspectrogram.onnx')
download(f'{BASE_RELEASE}/embedding_model.onnx', OWW_RESOURCES / 'embedding_model.onnx')

PIPER_VOICE_DIR = MODELS_DIR / 'piper'
PIPER_VOICE_DIR.mkdir(parents=True, exist_ok=True)
PIPER_BASE = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium'
download(f'{PIPER_BASE}/en_US-libritts_r-medium.pt', PIPER_MODEL_DRIVE)
download(f'{PIPER_BASE}/en_US-libritts_r-medium.pt.json', PIPER_CONFIG_DRIVE)

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
download('https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy', FEATURES_DIR / 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy')
download('https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy', FEATURES_DIR / 'validation_set_features.npy')

# Fondos: FMA y MIT RIR. No usar AudioSet.
archives_dir = DATASETS_DIR / '_archives'
archives_dir.mkdir(parents=True, exist_ok=True)
if len(list(FMA_PATH.glob('*.wav'))) < 120:
    fma_zip = download('https://os.unil.cloud.switch.ch/fma/fma_small.zip', archives_dir / 'fma_small.zip')
    extract_dir = archives_dir / 'fma_small_extract'
    if not extract_dir.exists():
        with zipfile.ZipFile(fma_zip) as zf:
            zf.extractall(extract_dir)
    sources = sorted([p for p in extract_dir.rglob('*') if p.suffix.lower() in {'.mp3', '.wav', '.flac', '.ogg', '.m4a'}])[:120]
    if len(sources) < 120:
        raise RuntimeError(f'FMA insuficiente: {len(sources)} audios')
    for index, src in enumerate(sources, start=1):
        dest = FMA_PATH / f'fma_background_{index:03d}.wav'
        if not dest.exists() or dest.stat().st_size == 0:
            convert_audio(src, dest)
else:
    print('Reutilizando FMA WAV:', len(list(FMA_PATH.glob('*.wav'))))

for wav in sorted(FMA_PATH.glob('*.wav')):
    if not is_pcm16_mono_16k(wav):
        convert_audio(wav, wav)

if len(list(MIT_RIRS_PATH.glob('*.wav'))) < 270:
    rir_zip = download('https://mcdermottlab.mit.edu/Reverb/IRMAudio/Audio.zip', archives_dir / 'mit_rirs_audio.zip')
    extract_dir = archives_dir / 'mit_rirs_extract'
    if not extract_dir.exists():
        with zipfile.ZipFile(rir_zip) as zf:
            zf.extractall(extract_dir)
    sources = sorted(extract_dir.rglob('*.wav'))[:270]
    if len(sources) < 270:
        raise RuntimeError(f'MIT RIR insuficiente: {len(sources)} WAV')
    for index, src in enumerate(sources, start=1):
        dest = MIT_RIRS_PATH / f'mit_rir_{index:03d}.wav'
        if not dest.exists() or dest.stat().st_size == 0:
            convert_audio(src, dest)
else:
    print('Reutilizando MIT RIR WAV:', len(list(MIT_RIRS_PATH.glob('*.wav'))))

print('FMA_PATH=', FMA_PATH, len(list(FMA_PATH.glob('*.wav'))))
print('MIT_RIRS_PATH=', MIT_RIRS_PATH, len(list(MIT_RIRS_PATH.glob('*.wav'))))

In [ ]:
# 7. Crear my_model_v2.yaml
import yaml

target_phrase = ['oh ree on', 'o ree on', 'oree on', 'orion']
model_name = 'orion_v2'
n_samples = 1500
n_samples_val = 1000
steps = 10000
target_accuracy = 0.6
target_recall = 0.25
background_paths = [str(FMA_PATH), str(MIT_RIRS_PATH)]

CONFIG_PATH = BASE_DIR / 'my_model_v2.yaml'
config = {
    'target_phrase': target_phrase,
    'model_name': model_name,
    'output_dir': str(BASE_DIR / 'training_output'),
    'piper_sample_generator_path': str(PIPER_DIR),
    'n_samples': n_samples,
    'n_samples_val': n_samples_val,
    'steps': steps,
    'target_accuracy': target_accuracy,
    'target_recall': target_recall,
    'background_paths': background_paths,
    'background_paths_duplication_rate': [1 for _ in background_paths],
    'rir_paths': [str(MIT_RIRS_PATH)],
    'custom_negative_phrases': ['orient', 'oreo', 'onion', 'or i am', 'area on', 'oh really', 'turn on', 'online'],
    'tts_batch_size': 16,
    'augmentation_rounds': 1,
    'augmentation_batch_size': 16,
    'feature_data_files': {
        'negative': str(FEATURES_DIR / 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'),
    },
    'false_positive_validation_data_path': str(FEATURES_DIR / 'validation_set_features.npy'),
    'model_type': 'dnn',
    'layer_size': 64,
    'batch_n_per_class': 64,
    'max_negative_weight': 1500,
    'target_false_positives_per_hour': 0.5,
}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print(CONFIG_PATH)
print(CONFIG_PATH.read_text())

In [ ]:
# 8. Generar clips sinteticos (idempotente)
env = os.environ.copy()
env['PYTHONPATH'] = f"{OPENWAKEWORD_DIR}:{PIPER_DIR}:" + env.get('PYTHONPATH', '')
TRAIN_PY = OPENWAKEWORD_DIR / 'openwakeword' / 'train.py'
OUTPUT_MODEL_DIR = BASE_DIR / 'training_output' / model_name
expected_dirs = ['positive_train', 'positive_test', 'negative_train', 'negative_test']
if all((OUTPUT_MODEL_DIR / d).exists() and len(list((OUTPUT_MODEL_DIR / d).glob('*.wav'))) >= (n_samples if 'train' in d else n_samples_val) for d in expected_dirs):
    print('Clips existentes suficientes; se reutilizan.')
else:
    run([sys.executable, TRAIN_PY, '--training_config', CONFIG_PATH, '--generate_clips'], env=env)
print('Clips en', OUTPUT_MODEL_DIR)

In [ ]:
# 9. Verificar conteos y normalizar clips
def count_wavs(path):
    return len(list(Path(path).glob('*.wav')))

def normalize_clip_tree(directory):
    converted = 0
    for wav in sorted(Path(directory).glob('*.wav')):
        if not is_pcm16_mono_16k(wav):
            convert_audio(wav, wav)
            converted += 1
    return converted

counts = {
    'positive_train': count_wavs(OUTPUT_MODEL_DIR / 'positive_train'),
    'positive_test': count_wavs(OUTPUT_MODEL_DIR / 'positive_test'),
    'negative_train': count_wavs(OUTPUT_MODEL_DIR / 'negative_train'),
    'negative_test': count_wavs(OUTPUT_MODEL_DIR / 'negative_test'),
}
print(counts)
assert counts['positive_train'] >= 1500
assert counts['positive_test'] >= 1000
assert counts['negative_train'] >= 1500
assert counts['negative_test'] >= 1000

for name in expected_dirs:
    converted = normalize_clip_tree(OUTPUT_MODEL_DIR / name)
    print(name, 'normalizados:', converted)

In [ ]:
# 10. Aumentar clips y generar features (idempotente)
import numpy as np

feature_files = [
    OUTPUT_MODEL_DIR / 'positive_features_train.npy',
    OUTPUT_MODEL_DIR / 'positive_features_test.npy',
    OUTPUT_MODEL_DIR / 'negative_features_train.npy',
    OUTPUT_MODEL_DIR / 'negative_features_test.npy',
]

def valid_npy(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        arr = np.load(path, mmap_mode='r')
        return getattr(arr, 'size', 0) > 0
    except Exception:
        return False

for path in feature_files:
    if path.exists() and not valid_npy(path):
        print('Eliminando feature parcial:', path)
        path.unlink()

if all(valid_npy(path) for path in feature_files):
    print('Features existentes; se reutilizan.')
else:
    run([sys.executable, TRAIN_PY, '--training_config', CONFIG_PATH, '--augment_clips'], env=env)

missing_features = [str(path) for path in feature_files if not valid_npy(path)]
if missing_features:
    raise RuntimeError('Features incompletas:\n' + '\n'.join(missing_features))
for path in feature_files:
    print(path, path.stat().st_size)

In [ ]:
# 11. Entrenar modelo ONNX
ONNX_PATH = BASE_DIR / 'training_output' / f'{model_name}.onnx'
if not all(path.exists() and path.stat().st_size > 0 for path in feature_files):
    raise RuntimeError('No se entrena: faltan features completas')
if ONNX_PATH.exists() and ONNX_PATH.stat().st_size > 0:
    print('Modelo ONNX existente; se reutiliza:', ONNX_PATH)
else:
    run([sys.executable, TRAIN_PY, '--training_config', CONFIG_PATH, '--train_model'], env=env)
print('ONNX_PATH=', ONNX_PATH)

In [ ]:
# 12. Verificar que exista orion_v2.onnx
assert ONNX_PATH.exists(), f'No existe {ONNX_PATH}'
assert ONNX_PATH.stat().st_size > 0, f'Modelo vacio: {ONNX_PATH}'
print('OK:', ONNX_PATH, ONNX_PATH.stat().st_size, 'bytes')

In [ ]:
# 13. Copiar a carpeta persistente en Drive
FINAL_MODEL = EXPORT_DIR / 'orion_v2.onnx'
shutil.copy2(ONNX_PATH, FINAL_MODEL)
print('Modelo persistente:', FINAL_MODEL)

In [ ]:
# 14. Descargar modelo desde Colab
from google.colab import files
files.download(str(FINAL_MODEL))

In [ ]:
# 15. Recuperacion directa: normalizar WAV, generar features, entrenar y exportar
# Ejecuta esta celda despues de las celdas 1 a 8 si Colab se desconecto tras generar los 5000 clips.
for name in expected_dirs:
    directory = OUTPUT_MODEL_DIR / name
    count = count_wavs(directory)
    expected = n_samples if 'train' in name else n_samples_val
    if count < expected:
        raise RuntimeError(f'{name} tiene {count} WAV; esperado minimo {expected}')
    converted = normalize_clip_tree(directory)
    print(name, 'normalizados:', converted)

for path in feature_files:
    if path.exists() and not valid_npy(path):
        print('Eliminando feature parcial:', path)
        path.unlink()

if not all(valid_npy(path) for path in feature_files):
    run([sys.executable, TRAIN_PY, '--training_config', CONFIG_PATH, '--augment_clips'], env=env)

missing_features = [str(path) for path in feature_files if not valid_npy(path)]
if missing_features:
    raise RuntimeError('Features incompletas:\n' + '\n'.join(missing_features))

if not ONNX_PATH.exists() or ONNX_PATH.stat().st_size == 0:
    run([sys.executable, TRAIN_PY, '--training_config', CONFIG_PATH, '--train_model'], env=env)

assert ONNX_PATH.exists() and ONNX_PATH.stat().st_size > 0, f'No existe ONNX valido: {ONNX_PATH}'
FINAL_MODEL = EXPORT_DIR / 'orion_v2.onnx'
shutil.copy2(ONNX_PATH, FINAL_MODEL)
print('Modelo persistente:', FINAL_MODEL, FINAL_MODEL.stat().st_size, 'bytes')